# DoS (Denial of Service), R2L (Remote to Local), U2R (User to Root), and Probing Project

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# --- 1. DATA ACQUISITION (Updated URL & Fail-safe) ---
print("Step 1: Downloading NSL-KDD Dataset...")
# Using a reliable raw mirror for the NSL-KDD dataset
url = "https://raw.githubusercontent.com/zihangLin/NSL-KDD/master/KDDTrain%2B.txt"

columns = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
           "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
           "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
           "num_shells","num_access_files","num_outbound_cmds","is_host_login",
           "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
           "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
           "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
           "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
           "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
           "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty_level"]

try:
    df = pd.read_csv(url, names=columns)
    print(f"Successfully loaded {len(df)} rows from GitHub.")
except:
    print("External download failed. Generating synthetic data for local testing...")
    # This ensures the code still runs for demonstration if the network is blocked
    data = np.random.randint(0, 100, size=(1000, len(columns)))
    df = pd.DataFrame(data, columns=columns)
    df['label'] = np.random.choice(['normal', 'neptune', 'satan', 'ipsweep', 'back'], 1000)

df = df.drop('difficulty_level', axis=1)

# --- 2. ATTACK MAPPING ---
print("Step 2: Categorizing attacks...")
attack_map = {
    'normal': 'normal',
    'back': 'dos', 'land': 'dos', 'neptune': 'dos', 'pod': 'dos', 'smurf': 'dos', 'teardrop': 'dos', 'mailbomb': 'dos', 'apache2': 'dos', 'processtable': 'dos', 'udpstorm': 'dos',
    'ipsweep': 'probe', 'mscan': 'probe', 'nmap': 'probe', 'portsweep': 'probe', 'saint': 'probe', 'mscan': 'probe', 'satan': 'probe', 'saint': 'probe',
    'ftp_write': 'r2l', 'guess_passwd': 'r2l', 'imap': 'r2l', 'multihop': 'r2l', 'phf': 'r2l', 'spy': 'r2l', 'warezclient': 'r2l', 'warezmaster': 'r2l', 'sendmail': 'r2l', 'named': 'r2l', 'snmpgetattack': 'r2l', 'snmpguess': 'r2l', 'xlock': 'r2l', 'xsnoop': 'r2l', 'worm': 'r2l',
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r', 'perl': 'u2r', 'rootkit': 'u2r', 'httptunnel': 'u2r', 'ps': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r'
}

df['label'] = df['label'].map(attack_map).fillna('other')

# --- 3. PREPROCESSING ---
print("Step 3: Encoding and Scaling...")
le = LabelEncoder()
for col in ['protocol_type', 'service', 'flag']:
    df[col] = le.fit_transform(df[col].astype(str))

target_le = LabelEncoder()
df['label_encoded'] = target_le.fit_transform(df['label'])
num_classes = len(target_le.classes_)

X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --- 4. RANDOM FOREST ---
print("Step 4: Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# --- 5. NEURAL NETWORK ---
print("Step 5: Training Deep Learning Model...")
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

nn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
nn_model.fit(X_train, y_train_cat, epochs=5, batch_size=64, verbose=1)

# --- 6. VISUALIZATION ---
print("\n" + "="*40)
print("RESULTS AND PERFORMANCE")
print("="*40)

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_le.classes_, yticklabels=target_le.classes_)
plt.title('Intrusion Classification - Confusion Matrix')
plt.show()

# Feature Importance
importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]
plt.figure(figsize=(10, 5))
plt.barh(range(10), importances[indices], color='teal')
plt.yticks(range(10), [X.columns[i] for i in indices])
plt.title('Top 10 Security Features (Random Forest)')
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(y_test, rf_preds, target_names=target_le.classes_))